# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [ ]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

In [ ]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

In [ ]:
response.json()["choices"][0]["message"]["content"]

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [ ]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [3]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



API key found and looks good so far!


In [5]:
from openai import OpenAI

In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", max_tokens=400, messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"Here's a fun one:\n\n**A group of porcupines is called a prickle!**"

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [ ]:
requests.get("http://localhost:11434").content

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [ ]:
!ollama pull llama3.2:1b

In [8]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [22]:
# Get a fun fact

# response = ollama.chat.completions.create(model="gemma3:270m", max_completion_tokens=100, max_tokens=10 , messages=[{"role": "user", "content": "who is papacodes"}])

response = ollama.chat.completions.create(model="llama3.2:1b", max_completion_tokens=100, max_tokens=100, messages=[{"role": "user", "content": "Tell me a fun fact about modi& epstein files"}])

response.choices[0].message.content

"I can provide you with some interesting information about Modi and Epstein files, also known as MFACS or Mandatory Actuarial Court Records.\n\nThe New York Post has released thousands of private court records from the Manhattan District Attorney's office for free download. These records include documents such as the marriage certificates, divorce decrees, and property listings, detailing the relationships between the spouses and their close associates of Donald Trump, including his children James and Ivanka, Jared Kushner, Eric Schmidt, and Reena Epstein.\n\nOne"

In [ ]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
from bs4 import BeautifulSoup
import requests
import urllib3

# Disable SSL warnings when verify=False
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)# Let's try out this utility
def fetch_website_contents1(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers, verify=False)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[0:100_000]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [3]:

def giveSummary(url1, model1, client1, numOfOutputTokens):
    webcontent = fetch_website_contents1(url1)
    print("fetch successful")
    response = client1.chat.completions.create(model=model1, max_tokens=numOfOutputTokens, messages=[{"role": "user", "content": f"give me actual summary easily understandble for layman. Use contents from : {webcontent}"}])
    print("llm inferencing received")
    return response.choices[0].message.content

In [4]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [5]:
url2Fetch = "https://www.digitalocean.com/community/tutorials/an-introduction-to-networking-terminology-interfaces-and-protocols"
url1 = "https://edwarddonner.com"
ed = fetch_website_contents1(url2Fetch)
print(ed)

An Introduction to Networking Terminology, Interfaces, and Protocols | DigitalOcean

Blog
Docs
Get Support
Contact Sales
DigitalOcean
Products
Featured Products
Droplets
Scalable virtual machines
Kubernetes
Scale more effectively
Gradient™ AI Inference Cloud
Build and scale with AI
Cloudways
Managed cloud hosting
App Platform
Get apps to market faster
Managed Databases
Fully-managed database hosting
Compute
Droplets
Kubernetes
CPU-Optimized Droplets
Functions
App Platform
Gradient™ AI Inference Cloud
GPU Droplets
1-Click Models
Platform
Bare Metal GPUs
Backups & Snapshots
Backups
Snapshots
SnapShooter
Networking
Virtual Private Cloud (VPC)
Partner Network Connect
Cloud Firewalls
Load Balancers
DNS
DDoS Protection
Managed Databases
MongoDB
Kafka
MySQL
PostgreSQL
Valkey
OpenSearch
Storage
Spaces Object Storage
Volume Block Storage
Network File Storage
Developer Tools
API
CLI
Support Plans
Monitoring
Uptime
Identity and Access Management
Marketplace
Droplet 1-Click
Kubernetes 1-Click
AI 1

In [6]:
display(Markdown(ed))

An Introduction to Networking Terminology, Interfaces, and Protocols | DigitalOcean

Blog
Docs
Get Support
Contact Sales
DigitalOcean
Products
Featured Products
Droplets
Scalable virtual machines
Kubernetes
Scale more effectively
Gradient™ AI Inference Cloud
Build and scale with AI
Cloudways
Managed cloud hosting
App Platform
Get apps to market faster
Managed Databases
Fully-managed database hosting
Compute
Droplets
Kubernetes
CPU-Optimized Droplets
Functions
App Platform
Gradient™ AI Inference Cloud
GPU Droplets
1-Click Models
Platform
Bare Metal GPUs
Backups & Snapshots
Backups
Snapshots
SnapShooter
Networking
Virtual Private Cloud (VPC)
Partner Network Connect
Cloud Firewalls
Load Balancers
DNS
DDoS Protection
Managed Databases
MongoDB
Kafka
MySQL
PostgreSQL
Valkey
OpenSearch
Storage
Spaces Object Storage
Volume Block Storage
Network File Storage
Developer Tools
API
CLI
Support Plans
Monitoring
Uptime
Identity and Access Management
Marketplace
Droplet 1-Click
Kubernetes 1-Click
AI 1-Click Models
Add-Ons
Cloud Website Hosting
Cloudways
See all products
Solutions
AI and Machine Learning
Develop, train, and deploy AI apps
GPUs
Platform
1-Click Models
HR Knowledge Assistant
Code Copilot
Support Ticket Triage
Recommendation Engine
Blockchain
Infrastructure for decentralized apps
Blogs, Forums and Content Websites
Lightning-fast, reliable CMS hosting
Wordpress
Ghost
Mastodon
Data Analytics
Real-time data processing at scale
Data Streaming
AdTech & Martech
Kafka
Developer Tools
DevOps and CI/CD solutions
CI/CD
Prototyping
Digital Marketing Agencies
Power your clients’ websites and campaigns
Freelancer
IT Consulting
Ecommerce
Build beautiful online storefronts
Dropshipping
WooCommerce
Magento
Game Development
Low-latency multiplayer servers
Minecraft Hosting
IoT
Connect to the power of the cloud
Kafka
ISVs
Streamlined ISV application development
Secure Web Hosting
Powerful protection from DDoS and more
Private VPN
Startup Cloud Hosting
Scalable, cost-effective infrastructure
Small Business
Video Streaming
High-bandwidth, low-latency delivery
Kafka
Web and Mobile Apps
Simple cross-platform app hosting
cPanel
Docker
Next.js
Node.js
Website Hosting
Fast page loads and reliable site uptime
VPS Hosting
Virtual Machines
Get help
Migration Assistance
Talk to an expert
See all solutions
Developers
Our Community
Community Home
DevOps and development guides
CSS-Tricks
All things web design
The Wave
Content to level up your business.
Resources
Tutorials
Questions and Answers
Marketplace
Tools
Write for DOnations
Customer Stories
DigitalOcean Blog
Pricing Calculator
Get Involved
DigitalOcean Startups
Open Source Sponsorships
Hacktoberfest
Deploy 2025
Wavemakers Program
Documentation
Quickstart
Compute
Gradient™ AI Platform
Storage
Managed Databases
Containers
Billing
API Reference
Partners
DigitalOcean Partner Programs
Become a Partner
Partner Services Program
DigitalOcean AI Partner Program
Marketplace
DigitalOcean Startups
Connect with a Partner
Partner Programs Resources
Customer Stories
DigitalOcean Onboarding Series
Training for Agencies and Freelancers
Price Estimate Calculator
Featured Partner Articles
Cloud cost optimization best practices
Read more
How to choose a cloud provider
Read more
DigitalOcean vs. AWS Lightsail: Which Cloud Platform is Right for You?
Read more
Questions?
Talk to an expert
Pricing
Log in
Log in to:
Community
DigitalOcean
Sign up
Sign up for:
Community
DigitalOcean
Blog
Docs
Get Support
Contact Sales
Log in
Log in to:
Community
DigitalOcean
Sign up
Sign up for:
Community
DigitalOcean
Tutorials
Questions
Product Docs
Search Community
Report this
What is the reason for this report?
This undefined is spam
This undefined is offensive
This undefined is off-topic
This undefined is other
Submit
Table of contents
Networking Glossary
Network Layers
Interfaces
Protocols
Conclusion
Tutorials
Linux Basics
An Introduction to Networking Terminology, Interfaces, and Protocols
Tutorial
An Introduction to Networking Terminology, Interfaces, and Protocols
Updated on October 4, 2022
Linux Basics
Networking
Conceptual
By
Justin Ellingwood
Table of  contents
Popular topics
Introduction
An understanding of networking is important for anyone managing a server. Not only is it essential for getting your services online and running smoothly, it also gives you the insight to diagnose problems.
This article will provide an overview of some common networking concepts. We will discuss terminology, common protocols, and the responsibilities and characteristics of the different layers of networking.
This guide is operating system agnostic, but should be very helpful when implementing features and services that utilize networking on your server.
Networking Glossary
First, we will define some common terms that you will see throughout this guide, and in other guides and documentation regarding networking.
These terms will be expanded upon in the appropriate sections that follow:
Connection
: In networking, a connection refers to pieces of related information that are transferred through a network. Generally speaking, a connection is established before data transfer (by following the procedures laid out in a protocol) and may be deconstructed at the end of the data transfer.
Packet
: A packet is the smallest unit that is intentionally transferred over a network. When communicating over a network, packets are the envelopes that carry your data (in pieces) from one end point to the other.
Packets have a header portion that contains information about the packet including the source and destination, timestamps, network hops, etc. The main portion of a packet contains the actual data being transferred. It is sometimes called the body or the payload.
Network Interface
: A network interface can refer to any kind of software interface to networking hardware. For instance, if you have two network cards in your computer, you can control and configure each network interface associated with them individually.
A network interface may be associated with a physical device, or it may be a representation of a virtual interface. The “loopback” device, which is a virtual interface available in most Linux environments to connect back to the same machine, is an example of this.
LAN
: LAN stands for “local area network”. It refers to a network or a portion of a network that is not publicly accessible to the greater internet. A home or office network is an example of a LAN.
WAN
: WAN stands for “wide area network”. It means a network that is much more extensive than a LAN. While WAN is the relevant term to use to describe large, dispersed networks in general, it is usually meant to mean the internet, as a whole.
If an interface is said to be connected to the WAN, it is generally assumed that it is reachable through the internet.
Protocol
: A protocol is a set of rules and standards that define a language that devices can use to communicate. There are a great number of protocols in use extensively in networking, and they are often implemented in different layers.
Some low level protocols are TCP, UDP, IP, and ICMP. Some familiar examples of application layer protocols, built on these lower protocols, are HTTP (for accessing web content), SSH, and TLS/SSL.
Port
: A port is an address on a single machine that can be tied to a specific piece of software. It is not a physical interface or location, but it allows your server to be able to communicate using more than one application.
Firewall
: A firewall is a program that decides whether traffic coming or going from a server should be allowed. A firewall usually works by creating rules for which type of traffic is acceptable on which ports. Generally, firewalls block ports that are not used by a specific application on a server.
NAT
: NAT stands for network address translation. It is a way to repackage and send incoming requests to a routing server to the relevant devices or servers on a LAN. This is usually implemented in physical LANs as a way to route requests through one IP address to the necessary backend servers.
VPN
: VPN stands for virtual private network. It is a means of connecting separate LANs through the internet, while maintaining privacy. This is used to connect remote systems as if they were on a local network, often for security reasons.
There are many other terms that you will come across, and this list is not exhaustive. We will explain other terms as we need them. At this point, you should understand some high-level concepts that will enable us to better discuss the topics to come.
Network Layers
While networking is often discussed in terms of topology in a horizontal way, between hosts, its implementation is layered in a vertical fashion within any given computer or network.
What this means is that there are multiple technologies and protocols that are built on top of each other in order for communication to function. Each successive, higher layer abstracts the raw data a little bit more.
It also allows you to leverage lower layers in new ways without having to invest the time and energy to develop the protocols and applications that handle those types of traffic.
The language that we use to talk about each of the layering schemes varies significantly depending on which model you use. Regardless of the model used to discuss the layers, the path of data is the same.
As data is sent out of one machine, it begins at the top of the stack and filters downwards. At the lowest level, actual transmission to another machine takes place. At this point, the data travels back up through the layers of the other computer.
Each layer has the ability to add its own “wrapper” around the data that it receives from the adjacent layer, which will help the layers that come after decide what to do with the data when it is handed off.
TCP/IP Model
The TCP/IP model, more commonly known as the Internet protocol suite, is a widely adopted layering model. It defines the four separate layers:
Application
:  In this model, the application layer is responsible for creating and transmitting user data between applications. The applications can be on remote systems, and should appear to operate as if locally to the end user. This communication is said to take place between
peers
.
Transport
:  The transport layer is responsible for communication between processes. This level of networking utilizes ports to address different services.
Internet
:  The internet layer is used to transport data from node to node in a network. This layer is aware of the endpoints of the connections, but is not concerned with the actual connection needed to get from one place to another.
IP addresses
are defined in this layer as a way of reaching remote systems in an addressable manner.
Link
:  The link layer implements the actual topology of the local network that allows the internet layer to present an addressable interface. It establishes connections between neighboring nodes to send data.
As you can see, the TCP/IP model is abstract and fluid. This made it popular to implement and allowed it to become the dominant way that networking layers are categorized.
Interfaces
Interfaces are networking communication points for your computer. Each interface is associated with a physical or virtual networking device.
Typically, your server will have one configurable network interface for each Ethernet or wireless internet card you have.
In addition, it will define a virtual network interface called the “loopback” or localhost interface. This is used as an interface to connect applications and processes on a single computer to other applications and processes. You can see this referenced as the “lo” interface in many tools.
Many times, administrators configure one interface to service traffic to the internet and another interface for a LAN or private network.
In datacenters with private networking enabled (including DigitalOcean Droplets), your VPS will have two networking interfaces. The “eth0” interface will be configured to handle traffic from the internet, while the “eth1” interface will operate to communicate with a private network.
Protocols
Networking works by piggybacking a number of different protocols on top of each other. In this way, one piece of data can be transmitted using multiple protocols encapsulated within one another.
We will start with protocols implemented on the lower networking layers and work our way up to protocols with higher abstraction.
Medium Access Control
Medium access control is a communications protocol that is used to distinguish specific devices. Each device is supposed to get a unique, hardcoded
media access control address
(MAC address) when it is manufactured that differentiates it from every other device on the internet.
Addressing hardware by the MAC address allows you to reference a device by a unique value even when the software on top may change the name for that specific device during operation.
MAC addressing is one of the only protocols from the low-level link layer that you are likely to interact with on a regular basis.
IP
The IP protocol is one of the fundamental protocols that allow the internet to work. IP addresses are unique on each network and they allow machines to address each other across a network. It is implemented on the internet layer in the TCP/IP model.
Networks can be linked together, but traffic must be routed when crossing network boundaries. This protocol assumes an unreliable network and multiple paths to the same destination that it can dynamically change between.
There are a number of different implementations of the protocol. The most common implementation today is IPv4 addresses, which follow the pattern
123.123.123.123
, although IPv6 addresses, which follows the pattern
2001:0db8:0000:0000:0000:ff00:0042:8329
, are growing in popularity due to the limited number of available IPv4 addresses.
ICMP
ICMP stands for internet control message protocol. It is used to send messages between devices to indicate their availability or error conditions. These packets are used in a variety of network diagnostic tools, such as
ping
and
traceroute
.
Usually ICMP packets are transmitted when a different kind of packet encounters a problem. They are used as a feedback mechanism for network communications.
TCP
TCP stands for transmission control protocol. It is implemented in the transport layer of the TCP/IP model and is used to establish reliable connections.
TCP is one of the protocols that encapsulates data into packets. It then transfers these to the remote end of the connection using the methods available on the lower layers. On the other end, it can check for errors, request certain pieces to be resent, and reassemble the information into one logical piece to send to the application layer.
The protocol builds up a connection prior to data transfer using a system called a three-way handshake. This is a way for the two ends of the communication to acknowledge the request and agree upon a method of ensuring data reliability.
After the data has been sent, the connection is torn down using a similar four-way handshake.
TCP is the protocol of choice for many of the most popular uses for the internet, including WWW, SSH, and email.
UDP
UDP stands for user datagram protocol. It is a popular companion protocol to TCP and is also implemented in the transport layer.
The fundamental difference between UDP and TCP is that UDP offers unreliable data transfer. It does not verify that data has been received on the other end of the connection. This might sound like a bad thing, and for many purposes, it is. However, it is also extremely important for some functions.
Because it is not required to wait for confirmation that the data was received and forced to resend data, UDP is much faster than TCP. It does not establish a connection with the remote host, it just sends data without confirmation.
Because it is a straightforward transaction, it is useful for communications like querying for network resources. It also doesn’t maintain a state, which makes it great for transmitting data from one machine to many real-time clients. This makes it ideal for VOIP, games, and other applications that cannot afford delays.
HTTP
HTTP stands for hypertext transfer protocol. It is a protocol defined in the application layer that forms the basis for communication on the web.
HTTP defines a number of verbs that tell the remote system what you are requesting. For instance, GET, POST, and DELETE all interact with the requested data in a different way. To see an example of the different HTTP requests in action, refer to
How To Define Routes and HTTP Request Methods in Express
.
DNS
DNS stands for domain name system. It is an application layer protocol used to provide a human-friendly naming mechanism for internet resources. It is what ties a domain name to an IP address and allows you to access sites by name in your browser.
SSH
SSH stands for secure shell. It is an encrypted protocol implemented in the application layer that can be used to communicate with a remote server in a secure way. Many additional technologies are built around this protocol because of its end-to-end encryption and ubiquity.
There are many other protocols that we haven’t covered that are equally important. However, this should give you a good overview of some of the fundamental technologies that make the internet and networking possible.
Conclusion
At this point, you should be familiar with some networking terminology and be able to understand how different components are able to communicate with each other. This should assist you in understanding other articles and the documentation of your system.
Next, for a high-level, read world example, you may want to read
How To Make HTTP Requests in Go
.
Thanks for learning with the DigitalOcean Community. Check out our offerings for compute, storage, networking, and managed databases.
Learn more about our products
About the author
Justin Ellingwood
Author
See author profile
Former Senior Technical Writer at DigitalOcean, specializing in DevOps topics across multiple Linux distributions, including Ubuntu 18.04, 20.04, 22.04, as well as Debian 10 and 11.
See author profile
Category:
Tutorial
Tags:
Linux Basics
Networking
Conceptual
Still looking for an answer?
Ask a question
Search for more help
Was this helpful?
Yes
No
Comments(10)
Follow-up questions(0)
﻿
This textbox defaults to using
Markdown
to format your answer.
You can type
!ref
in this text area to quickly search our full set of
tutorials, documentation & marketplace offerings and insert the link!
Sign in/up to comment
79a7fcaab293df33f44363a1608e4d
September 10, 2014
Show less
Thank You
Reply
veerathneetha
January 31, 2015
Show less
Define the network terminology, server. network interface card.
Reply
kryptoninho
August 21, 2015
Show less
this
is amaizing dude
******
Reply
nilufar
August 26, 2015
Show less
Thanks a lot for this article. Very clear and easy-to-understand. don’t you mind if i will take some definitions of texts from this article and create my own post and share it under my authority?
Reply
arulrajalivi
November 13, 2015
Show less
wonderfully written article about  the  fundamentals of networking.
Reply
papito
October 28, 2016
This comment has been deleted
bcb1000000
October 28, 2016
Show less
What is network terminology? Explain me sr
Reply
Davidzagi93
November 10, 2016
Show less
This resource was helpful. Thanks.
Reply
papito
November 11, 2016
This comment has been deleted
heshamaboelmagd
January 26, 2018
Show less
TCP is the protocol of choice for many of the most popular uses for the internet, including WWW, FTP, SSH, and email. It is safe to say that the internet we know today would not be here without TCP.
WWW = HTTP
Reply
Load more comments
This work is licensed under a Creative Commons Attribution-NonCommercial- ShareAlike 4.0 International License.
Deploy on DigitalOcean
Click below to sign up for DigitalOcean's virtual machines, Databases, and AIML products.
Sign up
Popular Topics
AI/ML
Ubuntu
Linux Basics
JavaScript
Python
MySQL
Docker
Kubernetes
All tutorials
Talk to an expert
Featured tutorials
SOLID Design Principles Explained: Building Better Software Architecture
How To Remove Docker Images, Containers, and Volumes
How to Create a MySQL User and Grant Privileges (Step-by-Step)
All tutorials
All topic tags
Join the Tech Talk
Success!
Thank you! Please check your email for further details.
Please complete your information!
Table of contents
Networking Glossary
Network Layers
Interfaces
Protocols
Conclusion
Deploy on DigitalOcean
Click below to sign up for DigitalOcean's virtual machines, Databases, and AIML products.
Sign up
Popular Topics
AI/ML
Ubuntu
Linux Basics
JavaScript
Python
MySQL
Docker
Kubernetes
All tutorials
Talk to an expert
Featured tutorials
SOLID Design Principles Explained: Building Better Software Architecture
How To Remove Docker Images, Containers, and Volumes
How to Create a MySQL User and Grant Privileges (Step-by-Step)
All tutorials
All topic tags
Become a contributor for community
Get paid to write technical tutorials and select a tech-focused charity to receive a matching donation.
Sign Up
DigitalOcean Documentation
Full documentation for every DigitalOcean product.
Learn more
Resources for startups and AI-native businesses
The Wave has everything you need to know about building a business, from raising funding to marketing your product.
Learn more
Get our newsletter
Stay up to date by signing up for DigitalOcean’s Infrastructure as a Newsletter.
Submit
Submit
New accounts only. By submitting your email you agree to our
Privacy Policy
The developer cloud
Scale up as you grow — whether you're running one virtual machine or ten thousand.
View all products
Get started for free
Sign up and get $200 in credit for your first 60 days with DigitalOcean.*
Get started
*This promotional offer applies to new accounts only.
Company
About
Leadership
Blog
Careers
Customers
Partners
Referral Program
Affiliate Program
Press
Legal
Privacy Policy
Security
Investor Relations
Products
Overview
Droplets
Kubernetes
Functions
App Platform
Gradient™ AI GPU Droplets
Gradient™ AI Bare Metal GPUs
Gradient™ AI 1-Click Models
Gradient™ AI Platform
Load Balancers
Managed Databases
Spaces
Block Storage
Network File Storage
API
Uptime
Identity and Access Management
Cloudways
Resources
Community Tutorials
Community Q&A
CSS-Tricks
Write for DOnations
Currents Research
DigitalOcean Startups
Wavemakers Program
Compass Council
Open Source
Newsletter Signup
Marketplace
Pricing
Pricing Calculator
Documentation
Release Notes
Code of Conduct
Shop Swag
Solutions
Website Hosting
VPS Hosting
Web & Mobile Apps
Game Development
Streaming
VPN
SaaS Platforms
Cloud Hosting for Blockchain
Startup Resources
Migration Assistance
Contact
Support
Sales
Report Abuse
System Status
Share your ideas
Company
About
Leadership
Blog
Careers
Customers
Partners
Referral Program
Affiliate Program
Press
Legal
Privacy Policy
Security
Investor Relations
Products
Overview
Droplets
Kubernetes
Functions
App Platform
Gradient™ AI GPU Droplets
Gradient™ AI Bare Metal GPUs
Gradient™ AI 1-Click Models
Gradient™ AI Platform
Load Balancers
Managed Databases
Spaces
Block Storage
Network File Storage
API
Uptime
Identity and Access Management
Cloudways
Resources
Community Tutorials
Community Q&A
CSS-Tricks
Write for DOnations
Currents Research
DigitalOcean Startups
Wavemakers Program
Compass Council
Open Source
Newsletter Signup
Marketplace
Pricing
Pricing Calculator
Documentation
Release Notes
Code of Conduct
Shop Swag
Solutions
Website Hosting
VPS Hosting
Web & Mobile Apps
Game Development
Streaming
VPN
SaaS Platforms
Cloud Hosting for Blockchain
Startup Resources
Migration Assistance
Contact
Support
Sales
Report Abuse
System Status
Share your ideas
©
2026
DigitalOcean, LLC.
Sitemap
.

In [7]:
ans = giveSummary(url2Fetch, "llama3.2:1b", ollama, 400)
display(Markdown(ans))

fetch successful
llm inferencing received


To summarize this text with respect to the question asked (Network Terminology), here are key points:

**Network Terminology:**

1. **Server**: A computer that provides services or resources over a network.
2. **Network Interface Card (NIC)**: The card on the back of a computer or device that connects it to a physical network, such as a LAN or WAN.

There is no specific definition provided for "network terminology" in this text. It appears to be a general overview of networking concepts and terminology used throughout various online platforms, including Cloudways, DigitalOcean, and Stack Overflow.

However, if we were to infer some terms that might be part of what network technicians use, they are:

1. **Router**: A device that connects multiple networks together.
2. ** Firewall**: A security system that controls incoming and outgoing network traffic based on predetermined rules.
3. ** DNS (Domain Name System)**: An application-layer protocol used to translate human-readable domain names into IP addresses.

Please note that these terms are commonly used in networking concepts, but the specific meanings might require additional context or explanation.

In [9]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)


In [10]:
#using gemini
# model=
gemini_ans = giveSummary(url2Fetch, "gemini-2.5-flash-lite", gemini, 400)
display(Markdown(gemini_ans))

fetch successful
llm inferencing received


Imagine the internet like a giant postal service. Here's a simple breakdown of the terms from the DigitalOcean article:

**How the Internet Talks (Networking Terminology):**

*   **Connection:** Think of this as a phone call or a specific conversation between two people. Before you can talk, you establish a connection. In networking, it's a way for devices to set up a line of communication.
*   **Packet:** Imagine your message is too big to fit on one postcard. You break it into smaller pieces, and each piece is a "packet." Each packet has an address on it (like a sender and receiver) and contains a part of your message.
*   **Network Interface:** This is like the "doorway" or "slot" on your computer that lets it connect to a network. Your computer might have physical ones (like a port for an Ethernet cable) or virtual ones.
*   **LAN (Local Area Network):** This is like your home or office network. It's a small, private network where devices can talk to each other easily.
*   **WAN (Wide Area Network):** This is a huge network that connects many LANs. The internet itself is the biggest WAN.
*   **Protocol:** These are like the languages and rules for how devices communicate. Just like you need to speak the same language as someone to understand them, devices need to follow the same protocols.
*   **Port:** On a single computer, this is like a specific extension number for different applications. If your computer is a building, ports are the different office doors where specific programs can receive messages.
*   **Firewall:** This is like a security guard at the entrance of your building (your computer). It checks who is allowed in and out, blocking unwanted traffic based on rules.
*   **NAT (Network Address Translation):** Imagine you have many devices in your house (your LAN